# **Person Dataset**

* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Person/f55k-p6yu/data_preview

In [51]:
import pandas as pd
from helpers import summarize_columns

df = pd.read_csv("datasets/Motor_Vehicle_Collisions_-_Person_20260424.csv", sep=",")
collIDs_df = pd.read_csv("datasets/collision_ids.csv", sep=",")

summarize_columns(df)

/tmp/ipykernel_14745/2351309123.py:4: DtypeWarning: Columns (0: PERSON_AGE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasets/Motor_Vehicle_Collisions_-_Person_20260424.csv", sep=",")


                     name    dtype   unique  size (MB)
0               UNIQUE_ID    int64  5942295         45
1            COLLISION_ID    int64  1626742         45
2              CRASH_DATE      str     5042        102
3              CRASH_TIME      str     1440         72
4               PERSON_ID      str  5747473        221
5             PERSON_TYPE      str        4         91
6           PERSON_INJURY      str        3        104
7              VEHICLE_ID  float64  2759037         45
8              PERSON_AGE   object     1060        271
9                EJECTION      str        7         78
10       EMOTIONAL_STATUS      str        9         85
11          BODILY_INJURY      str       15         85
12    POSITION_IN_VEHICLE      str       12        117
13       SAFETY_EQUIPMENT      str       18         86
14           PED_LOCATION      str        5         51
15             PED_ACTION      str       17         48
16              COMPLAINT      str       22         91
17        

#### <span style="background-color: RebeccaPurple;"> `Filtering-in COLLISION IDs present in the Crash Dataset` </span>

In [52]:
df = df[df["COLLISION_ID"].isin(collIDs_df["COLLISION_ID"])]
summarize_columns(df)

                     name    dtype   unique  size (MB)
0               UNIQUE_ID    int64  5344373         81
1            COLLISION_ID    int64  1473702         81
2              CRASH_DATE      str     5025        133
3              CRASH_TIME      str     1440        106
4               PERSON_ID      str  5179194        242
5             PERSON_TYPE      str        4        123
6           PERSON_INJURY      str        3        135
7              VEHICLE_ID  float64  2493335         81
8              PERSON_AGE   object     1005        285
9                EJECTION      str        7        110
10       EMOTIONAL_STATUS      str        9        117
11          BODILY_INJURY      str       15        117
12    POSITION_IN_VEHICLE      str       12        145
13       SAFETY_EQUIPMENT      str       18        117
14           PED_LOCATION      str        5         87
15             PED_ACTION      str       17         84
16              COMPLAINT      str       22        122
17        

## **Cleaning up section & Formatting**

In [53]:
# --- Person Age --- #
def _clean_person_age(df, col="PERSON_AGE"):
    """Setting ages outside the [0, 120] range to <NA>, allowing for Int8 dtype"""
    df[col] = df[col].str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    mask_invalid = (df[col] < 0) | (df[col] > 120)
    df.loc[mask_invalid, col] = pd.NA

    print(
        f"\t-> {sum(mask_invalid.notna() & mask_invalid)} invalid PERSON_AGE where  set to <NA> "
    )
    return df


df = _clean_person_age(df)


# --- Ped Location i.e. PED_LOCATION ---#
def _binarize_ped_location(df, col="PED_LOCATION"):
    """PED_LOCATION contains both <NA> and 'Unspecified', also naming is a bit too verbose for the entries"""
    d = {
        "Pedestrian/Bicyclist/Other Pedestrian at Intersection": "At Intersection",
        "Pedestrian/Bicyclist/Other Pedestrian Not at Intersection": "Not At Intersection",
    }
    df[col] = df[col].replace(d)

    print(
        f"\t-> {sum(~df[col].isin(d.values()) & df[col].notna())} 'Unspecified' entries were set to <NA>"
    )
    df.loc[~df[col].isin(d.values()), col] = pd.NA

    return df


df = _binarize_ped_location(df)


#! --- Person ID --- #
df["PERSON_ID"] = pd.factorize(df["PERSON_ID"])[0]


df["VEHICLE_ID"] = pd.to_numeric(df["VEHICLE_ID"], errors="coerce").astype("Int64")


# --- Mergin Time Columns in one DateTime column --- #
df["CRASH_DATETIME"] = pd.to_datetime(
    df["CRASH_DATE"] + " " + df["CRASH_TIME"], format="%m/%d/%Y %H:%M"
)

df = df.drop(["CRASH_TIME", "CRASH_DATE"], axis=1)


summarize_columns(df)

	-> 4052 invalid PERSON_AGE where  set to <NA> 
	-> 6734 'Unspecified' entries were set to <NA>
                     name           dtype   unique  size (MB)
0               UNIQUE_ID           int64  5344373         81
1            COLLISION_ID           int64  1473702         81
2               PERSON_ID           int64  5179194         81
3             PERSON_TYPE             str        4        123
4           PERSON_INJURY             str        3        135
5              VEHICLE_ID           Int64  2493335         86
6              PERSON_AGE           Int64      122         86
7                EJECTION             str        7        110
8        EMOTIONAL_STATUS             str        9        117
9           BODILY_INJURY             str       15        117
10    POSITION_IN_VEHICLE             str       12        145
11       SAFETY_EQUIPMENT             str       18        117
12           PED_LOCATION             str        3         83
13             PED_ACTION           

## **Memory Optimization**

In [54]:
_column_set = set(df.columns)


numeric_columns = ["UNIQUE_ID", "COLLISION_ID", "VEHICLE_ID", "PERSON_ID"]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

df["PERSON_AGE"] = pd.to_numeric(df["PERSON_AGE"], errors="coerce").astype("Int8")

category_columns = [
    "PERSON_TYPE",
    "PERSON_INJURY",
    "EJECTION",
    "EMOTIONAL_STATUS",
    "BODILY_INJURY",
    "POSITION_IN_VEHICLE",
    "SAFETY_EQUIPMENT",
    "PED_LOCATION",
    "PED_ACTION",
    "COMPLAINT",
    "PED_ROLE",
    "CONTRIBUTING_FACTOR_1",
    "CONTRIBUTING_FACTOR_2",
    "PERSON_SEX",
]

df = df.astype({c: "category" for c in category_columns})


# --- Reorder columns ----
new_col_order_list = (
    numeric_columns + ["CRASH_DATETIME"] + ["PERSON_AGE"] + category_columns
)

assert _column_set == set(_column_set)

summarize_columns(df)

                     name           dtype   unique  size (MB)
0               UNIQUE_ID           Int64  5344373         86
1            COLLISION_ID           Int64  1473702         86
2               PERSON_ID           Int64  5179194         86
3             PERSON_TYPE        category        4         45
4           PERSON_INJURY        category        3         45
5              VEHICLE_ID           Int64  2493335         86
6              PERSON_AGE            Int8      122         50
7                EJECTION        category        7         45
8        EMOTIONAL_STATUS        category        9         45
9           BODILY_INJURY        category       15         45
10    POSITION_IN_VEHICLE        category       12         45
11       SAFETY_EQUIPMENT        category       18         45
12           PED_LOCATION        category        3         45
13             PED_ACTION        category       17         45
14              COMPLAINT        category       22         45
15      

In [55]:
df = df.reset_index(drop=True)
df.to_parquet("datasets/person_data.parquet", engine="pyarrow")

In [56]:
del df